# AI CUP 2026 VeriPromiseESG 改進版 Multi-Task Transformer

以官方 baseline 的多任務分類架構為基礎，加入以下改進：

1. Dropout regularization
2. Class-weighted loss
3. 標籤一致性後處理
4. 超參數調整
5. 替換中文預訓練語言模型
6. Head + Tail 長文本處理
7. ESG 類別資訊加入輸入文字




## Step 1. 安裝套件

In [ ]:
!pip install transformers torch scikit-learn pandas matplotlib seaborn -q
print("套件安裝完成")

套件安裝完成


## Step 2. 匯入套件與固定隨機種子

In [ ]:
import os
import json
import random
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from collections import Counter


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用裝置:", device)
print("PyTorch version:", torch.__version__)

使用裝置: cuda
PyTorch version: 2.11.0+cu128


## Step 3. 超參數設定

可從這裡調整模型與實驗設定。

In [ ]:
# ============================================================
# 模型選擇
# ============================================================
# 可嘗試：
# MODEL_NAME = "bert-base-chinese"
# MODEL_NAME = "hfl/chinese-roberta-wwm-ext"
# MODEL_NAME = "hfl/chinese-macbert-base"
# MODEL_NAME = "hfl/chinese-bert-wwm-ext"

MODEL_NAME = "hfl/chinese-macbert-base"

# ============================================================
# 訓練超參數
# ============================================================
MAX_LEN = 384
BATCH_SIZE = 8
EPOCHS = 8
LR = 2e-5
DROPOUT_RATE = 0.2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

# ============================================================
# 改進開關
# ============================================================
USE_CLASS_WEIGHT = True
USE_FIELD_LOSS_WEIGHT = True
USE_POSTPROCESS = True
USE_ESG_TYPE = True

# 長文本處理模式：
# "truncate"：只取前 MAX_LEN tokens
# "head_tail"：取前半段 + 後半段 tokens
TEXT_MODE = "head_tail"

print("MODEL_NAME:", MODEL_NAME)
print("MAX_LEN:", MAX_LEN)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS:", EPOCHS)
print("TEXT_MODE:", TEXT_MODE)
print("USE_CLASS_WEIGHT:", USE_CLASS_WEIGHT)
print("USE_POSTPROCESS:", USE_POSTPROCESS)
print("USE_ESG_TYPE:", USE_ESG_TYPE)

MODEL_NAME: hfl/chinese-macbert-base
MAX_LEN: 384
BATCH_SIZE: 8
EPOCHS: 8
TEXT_MODE: head_tail
USE_CLASS_WEIGHT: True
USE_POSTPROCESS: True
USE_ESG_TYPE: True


## Step 4. 下載與載入訓練資料

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/veripromiseesg/veripromiseesgdataset/refs/heads/main/vpesg4k_train_1000.json"
TRAIN_PATH = "vpesg4k_train_1000.json"

if not os.path.exists(TRAIN_PATH):
    urllib.request.urlretrieve(DATA_URL, TRAIN_PATH)
    print("資料下載完成")
else:
    print("資料已存在，略過下載")

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    all_data = json.load(f)

print("資料總筆數:", len(all_data))
print("第一筆資料欄位:", list(all_data[0].keys()))
print("第一筆文字預覽:", all_data[0]["data"][:120])

資料已存在，略過下載
資料總筆數: 1000
第一筆資料欄位: ['id', 'data', 'esg_type', 'promise_status', 'promise_string', 'verification_timeline', 'evidence_status', 'evidence_string', 'evidence_quality', 'company', 'ticker', 'page_number', 'pdf_url', 'company_source']
第一筆文字預覽: 聯發科技除在「工作規則」中依照勞基法明確規定「員工在產假期間公司不得終止勞動契約」外，為支持同仁與其家人度過人生不同階段，自 2024 年起提供女性員工在分娩前後計有 12 週共 84 天的產假；男性員工則可於其配偶懷孕期間陪同產檢或生（流


## Step 5. 標籤設定與資料切分

In [ ]:
EVAL_FIELDS = {
    "promise_status": ["Yes", "No"],
    "verification_timeline": ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years", "N/A"],
    "evidence_status": ["Yes", "No", "N/A"],
    "evidence_quality": ["Clear", "Not Clear", "Misleading", "N/A"]
}

# 競賽評分權重，也可用於 loss 權重
FIELD_WEIGHTS = {
    "promise_status": 0.2,
    "verification_timeline": 0.15,
    "evidence_status": 0.3,
    "evidence_quality": 0.35
}

label2id = {
    field: {label: i for i, label in enumerate(labels)}
    for field, labels in EVAL_FIELDS.items()
}

id2label = {
    field: {i: label for i, label in enumerate(labels)}
    for field, labels in EVAL_FIELDS.items()
}

num_labels = {
    field: len(labels)
    for field, labels in EVAL_FIELDS.items()
}

# 以 promise_status stratify，讓訓練/驗證的 Yes/No 比例較接近
train_data, val_data = train_test_split(
    all_data,
    test_size=0.2,
    random_state=42,
    stratify=[item["promise_status"] for item in all_data]
)

print("訓練集:", len(train_data))
print("驗證集:", len(val_data))
print("標籤數量:", num_labels)

訓練集: 800
驗證集: 200
標籤數量: {'promise_status': 2, 'verification_timeline': 5, 'evidence_status': 3, 'evidence_quality': 4}


## Step 6. EDA：觀察標籤分佈

In [ ]:
train_df = pd.DataFrame(train_data)

for field in EVAL_FIELDS:
    print("\n" + "=" * 60)
    print(field)
    print(train_df[field].value_counts())


promise_status
promise_status
Yes    651
No     149
Name: count, dtype: int64

verification_timeline
verification_timeline
already                  303
between_2_and_5_years    184
more_than_5_years        156
N/A                      149
within_2_years             8
Name: count, dtype: int64

evidence_status
evidence_status
Yes    542
N/A    149
No     109
Name: count, dtype: int64

evidence_quality
evidence_quality
Clear         445
N/A           258
Not Clear      96
Misleading      1
Name: count, dtype: int64


## Step 7. 建立輸入文字與 Head + Tail tokenizer

這裡加入兩個改進：

- 將 `esg_type` 加入輸入文字。
- 對長文本使用 Head + Tail，保留前段與後段資訊。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def build_input_text(item, use_esg_type=True):
    """將 ESG 類別資訊加入模型輸入。"""
    text = str(item.get("data", ""))

    if use_esg_type:
        esg_type = str(item.get("esg_type", "Unknown"))
        text = f"ESG類別：{esg_type}。文本：{text}"

    return text


def encode_text(text, tokenizer, max_len=384, mode="head_tail"):
    """
    穩定版 tokenizer function。
    不使用 tokenizer.num_special_tokens_to_add()
    不使用 tokenizer.build_inputs_with_special_tokens()
    """

    if mode == "truncate":
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True
        )
        return encoded["input_ids"], encoded["attention_mask"]

    if mode != "head_tail":
        raise ValueError("mode 必須是 'truncate' 或 'head_tail'")

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id

    if pad_id is None:
        pad_id = 0

    if cls_id is None or sep_id is None:
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True
        )
        return encoded["input_ids"], encoded["attention_mask"]

    available_len = max_len - 2

    if len(token_ids) > available_len:
        head_len = available_len // 2
        tail_len = available_len - head_len
        token_ids = token_ids[:head_len] + token_ids[-tail_len:]

    input_ids = [cls_id] + token_ids + [sep_id]
    attention_mask = [1] * len(input_ids)

    pad_len = max_len - len(input_ids)
    if pad_len > 0:
        input_ids += [pad_id] * pad_len
        attention_mask += [0] * pad_len

    input_ids = input_ids[:max_len]
    attention_mask = attention_mask[:max_len]

    return input_ids, attention_mask


# 測試 encode 是否正常
sample_text = build_input_text(train_data[0], USE_ESG_TYPE)
ids, mask = encode_text(sample_text, tokenizer, MAX_LEN, TEXT_MODE)
print("sample text:", sample_text[:120])
print("input_ids length:", len(ids))
print("attention_mask length:", len(mask))

sample text: ESG類別：S。文本：為持續提升公司競爭力與員工專業發展，依據 EMC 訓練體系，2024年度訓練以「強化職能發展，友善學習體驗」為計畫主軸，全面推動各項訓練專案。
input_ids length: 384
attention_mask length: 384


## Step 8. Dataset 與 DataLoader

In [ ]:
class VeriPromiseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len, label2id, use_esg_type=True, text_mode="head_tail"):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label2id = label2id
        self.use_esg_type = use_esg_type
        self.text_mode = text_mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = build_input_text(item, self.use_esg_type)

        input_ids, attention_mask = encode_text(
            text,
            tokenizer=self.tokenizer,
            max_len=self.max_len,
            mode=self.text_mode
        )

        labels = {}
        for field in EVAL_FIELDS:
            label = item[field]
            labels[field] = self.label2id[field][label]

        return {
            "id": item.get("id", idx),
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": {
                field: torch.tensor(label_id, dtype=torch.long)
                for field, label_id in labels.items()
            }
        }


train_dataset = VeriPromiseDataset(
    train_data,
    tokenizer,
    MAX_LEN,
    label2id,
    use_esg_type=USE_ESG_TYPE,
    text_mode=TEXT_MODE
)

val_dataset = VeriPromiseDataset(
    val_data,
    tokenizer,
    MAX_LEN,
    label2id,
    use_esg_type=USE_ESG_TYPE,
    text_mode=TEXT_MODE
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels keys:", batch["labels"].keys())

input_ids shape: torch.Size([8, 384])
attention_mask shape: torch.Size([8, 384])
labels keys: dict_keys(['promise_status', 'verification_timeline', 'evidence_status', 'evidence_quality'])


## Step 9. 計算 Class Weights

用來處理類別不平衡問題。

In [ ]:
def compute_class_weights(data, field, label2id):
    labels = [item[field] for item in data]
    counts = Counter(labels)

    weights = []
    total = len(labels)
    n_classes = len(label2id[field])

    for label in EVAL_FIELDS[field]:
        count = counts.get(label, 0)
        if count == 0:
            weight = 0.0
        else:
            weight = total / (n_classes * count)
        weights.append(weight)

    return torch.tensor(weights, dtype=torch.float)


class_weights = {}

for field in EVAL_FIELDS:
    class_weights[field] = compute_class_weights(train_data, field, label2id).to(device)
    print(field, class_weights[field].detach().cpu().numpy())

promise_status [0.6144393 2.6845639]
verification_timeline [ 0.5280528 20.         0.8695652  1.0256411  1.0738255]
evidence_status [0.49200493 2.4464831  1.7897092 ]
evidence_quality [  0.4494382   2.0833333 200.          0.7751938]


## Step 10. 改進版 Multi-Task Transformer Model

模型架構：

- 共享同一個 Transformer encoder。
- 使用 CLS token 作為句子表示。
- 加入 Dropout。
- 四個任務各自有一個分類器。

In [ ]:
class ImprovedMultiTaskTransformer(nn.Module):
    def __init__(self, model_name, num_labels, dropout_rate=0.2):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout_rate)

        self.classifiers = nn.ModuleDict({
            field: nn.Linear(hidden_size, n_labels)
            for field, n_labels in num_labels.items()
        })

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # 使用 CLS token 表示整句語意
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = {
            field: classifier(cls_output)
            for field, classifier in self.classifiers.items()
        }

        return logits


model = ImprovedMultiTaskTransformer(
    MODEL_NAME,
    num_labels,
    dropout_rate=DROPOUT_RATE
).to(device)

print(model.__class__.__name__)
print("模型建立完成")

pytorch_model.bin:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ImprovedMultiTaskTransformer
模型建立完成


## Step 11. Loss function

加入兩個改進：

- Class-weighted Cross Entropy Loss
- 根據競賽評分權重加權四個任務 loss

In [ ]:
def compute_multitask_loss(logits, labels):
    total_loss = 0.0

    for field in EVAL_FIELDS:
        y_true = labels[field].to(device)

        if USE_CLASS_WEIGHT:
            criterion = nn.CrossEntropyLoss(weight=class_weights[field])
        else:
            criterion = nn.CrossEntropyLoss()

        loss = criterion(logits[field], y_true)

        if USE_FIELD_LOSS_WEIGHT:
            loss = FIELD_WEIGHTS[field] * loss

        total_loss += loss

    return total_loss

## Step 12. 標籤一致性後處理

In [ ]:
def apply_label_consistency(pred_item):
    """
    規則：
    1. 如果沒有承諾，其他欄位應為 N/A。
    2. 如果沒有 evidence，evidence_quality 應為 N/A。
    """

    pred_item = pred_item.copy()

    if pred_item["promise_status"] == "No":
        pred_item["verification_timeline"] = "N/A"
        pred_item["evidence_status"] = "N/A"
        pred_item["evidence_quality"] = "N/A"

    if pred_item["evidence_status"] in ["No", "N/A"]:
        pred_item["evidence_quality"] = "N/A"

    return pred_item


# 測試後處理
example = {
    "promise_status": "No",
    "verification_timeline": "within_2_years",
    "evidence_status": "Yes",
    "evidence_quality": "Clear"
}
print("before:", example)
print("after :", apply_label_consistency(example))

before: {'promise_status': 'No', 'verification_timeline': 'within_2_years', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}
after : {'promise_status': 'No', 'verification_timeline': 'N/A', 'evidence_status': 'N/A', 'evidence_quality': 'N/A'}


## Step 13. 評估函式

In [ ]:
def predict_labels(model, dataloader, device, use_postprocess=True):
    model.eval()

    all_preds = []
    all_truths = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(input_ids, attention_mask)
            batch_size = input_ids.size(0)

            for i in range(batch_size):
                pred_item = {}
                truth_item = {}

                for field in EVAL_FIELDS:
                    pred_id = torch.argmax(logits[field][i]).item()
                    pred_label = id2label[field][pred_id]
                    pred_item[field] = pred_label

                    true_id = batch["labels"][field][i].item()
                    true_label = id2label[field][true_id]
                    truth_item[field] = true_label

                if use_postprocess:
                    pred_item = apply_label_consistency(pred_item)

                all_preds.append(pred_item)
                all_truths.append(truth_item)

    return all_preds, all_truths


def evaluate_model(model, dataloader, device, use_postprocess=True, verbose=True):
    preds, truths = predict_labels(
        model,
        dataloader,
        device,
        use_postprocess=use_postprocess
    )

    field_scores = {}

    for field in EVAL_FIELDS:
        y_true = [item[field] for item in truths]
        y_pred = [item[field] for item in preds]

        score = f1_score(
            y_true,
            y_pred,
            labels=EVAL_FIELDS[field],
            average="macro",
            zero_division=0
        )

        field_scores[field] = score

    final_score = sum(
        FIELD_WEIGHTS[field] * field_scores[field]
        for field in EVAL_FIELDS
    )

    if verbose:
        print("\n各任務 Macro-F1:")
        for field, score in field_scores.items():
            print(f"{field}: {score:.4f}")
        print(f"加權總分: {final_score:.4f}")

    return final_score, field_scores, preds, truths


def print_classification_reports(preds, truths):
    for field in EVAL_FIELDS:
        y_true = [item[field] for item in truths]
        y_pred = [item[field] for item in preds]

        print("\n" + "=" * 60)
        print(field)
        print("=" * 60)

        print(classification_report(
            y_true,
            y_pred,
            labels=EVAL_FIELDS[field],
            zero_division=0
        ))

## Step 14. Optimizer 與 Scheduler

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("total_steps:", total_steps)
print("warmup_steps:", warmup_steps)

total_steps: 800
warmup_steps: 80


## Step 15. 訓練模型

In [ ]:
best_score = -1
best_model_path = "best_improved_model.pt"

train_history = []
val_history = []
val_field_history = []

for epoch in range(EPOCHS):
    print("\n" + "=" * 70)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 70)

    model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        labels = {
            field: batch["labels"][field].to(device)
            for field in EVAL_FIELDS
        }

        logits = model(input_ids, attention_mask)
        loss = compute_multitask_loss(logits, labels)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f"Step {step + 1}/{len(train_loader)} - Loss: {loss.item():.4f}")

    avg_train_loss = total_loss / len(train_loader)
    train_history.append(avg_train_loss)

    print(f"\n平均訓練 Loss: {avg_train_loss:.4f}")

    val_score, val_field_scores, _, _ = evaluate_model(
        model,
        val_loader,
        device,
        use_postprocess=USE_POSTPROCESS,
        verbose=True
    )

    val_history.append(val_score)
    val_field_history.append(val_field_scores)

    if val_score > best_score:
        best_score = val_score
        torch.save(model.state_dict(), best_model_path)
        print(f"目前最佳模型已儲存，分數: {best_score:.4f}")

print("\n訓練完成")
print("最佳驗證分數:", best_score)


Epoch 1/8
Step 50/100 - Loss: 1.0752
Step 100/100 - Loss: 0.9867

平均訓練 Loss: 1.0988

各任務 Macro-F1:
promise_status: 0.7829
verification_timeline: 0.3953
evidence_status: 0.5412
evidence_quality: 0.3661
加權總分: 0.5063
目前最佳模型已儲存，分數: 0.5063

Epoch 2/8
Step 50/100 - Loss: 0.5394
Step 100/100 - Loss: 0.7865

平均訓練 Loss: 0.8623

各任務 Macro-F1:
promise_status: 0.7582
verification_timeline: 0.4175
evidence_status: 0.6020
evidence_quality: 0.3728
加權總分: 0.5254
目前最佳模型已儲存，分數: 0.5254

Epoch 3/8
Step 50/100 - Loss: 0.3382
Step 100/100 - Loss: 0.7134

平均訓練 Loss: 0.6414

各任務 Macro-F1:
promise_status: 0.7874
verification_timeline: 0.4185
evidence_status: 0.6115
evidence_quality: 0.3712
加權總分: 0.5336
目前最佳模型已儲存，分數: 0.5336

Epoch 4/8
Step 50/100 - Loss: 0.3053
Step 100/100 - Loss: 0.9098

平均訓練 Loss: 0.4840

各任務 Macro-F1:
promise_status: 0.7713
verification_timeline: 0.4614
evidence_status: 0.5933
evidence_quality: 0.4207
加權總分: 0.5487
目前最佳模型已儲存，分數: 0.5487

Epoch 5/8
Step 50/100 - Loss: 0.3492
Step 100/100 - Los

## Step 16. 載入最佳模型並輸出詳細報告

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))

best_score, field_scores, preds, truths = evaluate_model(
    model,
    val_loader,
    device,
    use_postprocess=USE_POSTPROCESS,
    verbose=True
)

print_classification_reports(preds, truths)


各任務 Macro-F1:
promise_status: 0.7582
verification_timeline: 0.4879
evidence_status: 0.6134
evidence_quality: 0.4504
加權總分: 0.5665

promise_status
              precision    recall  f1-score   support

         Yes       0.90      0.95      0.92       163
          No       0.70      0.51      0.59        37

    accuracy                           0.87       200
   macro avg       0.80      0.73      0.76       200
weighted avg       0.86      0.87      0.86       200


verification_timeline
                       precision    recall  f1-score   support

              already       0.53      0.67      0.59        63
       within_2_years       0.00      0.00      0.00         5
between_2_and_5_years       0.62      0.59      0.60        54
    more_than_5_years       0.64      0.66      0.65        41
                  N/A       0.70      0.51      0.59        37

             accuracy                           0.60       200
            macro avg       0.50      0.49      0.49       20

## Step 17. 儲存 validation 預測結果

In [ ]:
output_predictions = []

for original_item, pred_item, truth_item in zip(val_data, preds, truths):
    output_item = {
        "id": original_item.get("id"),
        "prediction": pred_item,
        "ground_truth": truth_item
    }
    output_predictions.append(output_item)

with open("improved_validation_predictions.json", "w", encoding="utf-8") as f:
    json.dump(output_predictions, f, ensure_ascii=False, indent=2)

summary = {
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "dropout_rate": DROPOUT_RATE,
    "text_mode": TEXT_MODE,
    "use_class_weight": USE_CLASS_WEIGHT,
    "use_field_loss_weight": USE_FIELD_LOSS_WEIGHT,
    "use_postprocess": USE_POSTPROCESS,
    "use_esg_type": USE_ESG_TYPE,
    "best_weighted_score": best_score,
    "field_scores": field_scores
}

with open("experiment_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("已儲存 improved_validation_predictions.json")
print("已儲存 experiment_summary.json")
print(json.dumps(summary, ensure_ascii=False, indent=2))

已儲存 improved_validation_predictions.json
已儲存 experiment_summary.json
{
  "model_name": "hfl/chinese-macbert-base",
  "max_len": 384,
  "batch_size": 8,
  "epochs": 8,
  "learning_rate": 2e-05,
  "dropout_rate": 0.2,
  "text_mode": "head_tail",
  "use_class_weight": true,
  "use_field_loss_weight": true,
  "use_postprocess": true,
  "use_esg_type": true,
  "best_weighted_score": 0.5664785960009552,
  "field_scores": {
    "promise_status": 0.7581845238095238,
    "verification_timeline": 0.4879350580637725,
    "evidence_status": 0.6134089205397302,
    "evidence_quality": 0.4503678753359015
  }
}


## Step 18. 可選：產生測試集預測檔

若你在 Colab 左側上傳 `vpesg4k_test_2000.json`，可以執行下面 cell 產生 `submission.json`。

注意：測試集通常沒有標籤，所以這個 Dataset 不會讀取 label。

In [ ]:
class VeriPromiseTestDataset(Dataset):
    def __init__(self, data, tokenizer, max_len, use_esg_type=True, text_mode="head_tail"):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.use_esg_type = use_esg_type
        self.text_mode = text_mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = build_input_text(item, self.use_esg_type)

        input_ids, attention_mask = encode_text(
            text,
            tokenizer=self.tokenizer,
            max_len=self.max_len,
            mode=self.text_mode
        )

        return {
            "id": item.get("id", idx),
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long)
        }


def predict_test(model, dataloader, device, use_postprocess=True):
    model.eval()
    outputs = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)

            batch_size = input_ids.size(0)
            for i in range(batch_size):
                pred_item = {}
                for field in EVAL_FIELDS:
                    pred_id = torch.argmax(logits[field][i]).item()
                    pred_item[field] = id2label[field][pred_id]

                if use_postprocess:
                    pred_item = apply_label_consistency(pred_item)

                output_item = {"id": str(batch["id"][i])}
                output_item.update(pred_item)
                outputs.append(output_item)

    return outputs


TEST_PATH = "vpesg4k_test_2000.json"

if os.path.exists(TEST_PATH):
    with open(TEST_PATH, "r", encoding="utf-8") as f:
        test_data = json.load(f)

    test_dataset = VeriPromiseTestDataset(
        test_data,
        tokenizer,
        MAX_LEN,
        use_esg_type=USE_ESG_TYPE,
        text_mode=TEXT_MODE
    )

    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    submission = predict_test(model, test_loader, device, use_postprocess=USE_POSTPROCESS)

    with open("submission.json", "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    print("已產生 submission.json")
    print("預測筆數:", len(submission))
    print("前 3 筆:")
    print(json.dumps(submission[:3], ensure_ascii=False, indent=2))
else:
    print(f"找不到 {TEST_PATH}。若要產生測試集預測，請先把 test json 上傳到 Colab 工作目錄。")

已產生 submission.json
預測筆數: 2000
前 3 筆:
[
  {
    "id": "12001",
    "promise_status": "Yes",
    "verification_timeline": "already",
    "evidence_status": "Yes",
    "evidence_quality": "Clear"
  },
  {
    "id": "12002",
    "promise_status": "Yes",
    "verification_timeline": "already",
    "evidence_status": "Yes",
    "evidence_quality": "Clear"
  },
  {
    "id": "12003",
    "promise_status": "Yes",
    "verification_timeline": "within_2_years",
    "evidence_status": "Yes",
    "evidence_quality": "Clear"
  }
]
